# ⚡ Flujos de Trabajo Concurrentes de Agentes con Modelos de GitHub (Python)

## 📋 Tutorial Avanzado de Procesamiento Paralelo

Este cuaderno demuestra **patrones de flujo de trabajo concurrentes** usando el Microsoft Agent Framework. Aprenderás a construir flujos de trabajo de procesamiento paralelo de alto rendimiento donde múltiples agentes de IA ejecutan simultáneamente, mejorando drásticamente el rendimiento y permitiendo procesos comerciales sofisticados y multihilo.

## 🎯 Objetivos de Aprendizaje

### 🚀 **Fundamentos del Procesamiento Concurrente**
- **Ejecución Paralela de Agentes**: Ejecutar múltiples agentes simultáneamente para máxima eficiencia
- **Orquestación del Flujo de Trabajo**: Coordinar operaciones concurrentes manteniendo la consistencia de datos
- **Optimización del Rendimiento**: Lograr una aceleración significativa mediante procesamiento paralelo
- **Gestión de Recursos**: Utilizar eficientemente los recursos del modelo de IA en operaciones concurrentes

### 🏗️ **Patrones Avanzados de Concurrencia**
- **Procesamiento Fork-Join**: Dividir el trabajo entre varios agentes y combinar resultados
- **Paralelismo en Pipeline**: Superposición de etapas de ejecución para un rendimiento continuo
- **Balanceo de Carga**: Distribuir el trabajo de manera uniforme entre los recursos de agentes disponibles
- **Puntos de Sincronización**: Coordinar agentes concurrentes en etapas críticas del flujo de trabajo

### 🏢 **Aplicaciones Empresariales Concurrentes**
- **Procesamiento de Documentos en Alto Volumen**: Procesar múltiples documentos simultáneamente
- **Análisis de Contenido en Tiempo Real**: Análisis concurrente de flujos de datos entrantes
- **Optimización de Procesamiento por Lotes**: Maximizar el rendimiento en operaciones a gran escala
- **Análisis Multimodal**: Procesamiento paralelo de diferentes tipos de contenido (texto, imágenes, datos)

## ⚙️ Requisitos Previos y Configuración

### 📦 **Dependencias Requeridas**

Instala Agent Framework con capacidades de flujo de trabajo concurrente:

```bash
pip install agent-framework-core -U
```

### 🔑 **Configuración de Modelos GitHub**

**Configuración del Entorno (archivo .env):**
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

**Consideraciones para Procesamiento Concurrente:**
- **Límites de Tasa**: Monitorea los límites de tasa de la API de Modelos GitHub para solicitudes concurrentes
- **Uso de Recursos**: Considera el uso de memoria y CPU con múltiples agentes concurrentes
- **Manejo de Errores**: Implementa recuperación robusta de errores para operaciones paralelas

### 🏗️ **Arquitectura de Flujo de Trabajo Concurrente**

```mermaid
graph TD
    A[Inicio del Flujo de Trabajo] --> B[Ejecución Concurrente]
    B --> C[Grupo de Agentes 1]
    B --> D[Grupo de Agentes 2]
    B --> E[Grupo de Agentes 3]
    C --> F[Agregación de Resultados]
    D --> F
    E --> F
    F --> G[Salida Final]
    
    H[API de Modelos de GitHub] --> C
    H --> D
    H --> E
```

**Beneficios Clave:**
- **⚡ Rendimiento**: Aceleración significativa mediante ejecución paralela
- **📈 Escalabilidad**: Maneja cargas de trabajo aumentadas sin incremento proporcional en tiempo
- **🔄 Eficiencia**: Mejor aprovechamiento de los recursos computacionales disponibles
- **🎯 Rendimiento Total**: Procesa más trabajo en la misma cantidad de tiempo

## 🎨 **Patrones de Diseño de Flujos de Trabajo Concurrentes**

### 🔍 **Pipeline de Investigación y Análisis**
```
Research Task → Parallel Research Agents → Content Synthesis → Quality Review
```

### 📊 **Flujo de Trabajo de Procesamiento de Datos**
```
Input Data → Concurrent Processing Agents → Result Aggregation → Final Report
```

### 🎭 **Pipeline de Creación de Contenido**
```
Content Brief → Parallel Content Generators → Review & Merge → Final Content
```

### 🔄 **Procesamiento en Múltiples Etapas**
```
Input → Stage 1 (Concurrent) → Stage 2 (Concurrent) → Stage 3 (Sequential) → Output
```

## 🏢 **Beneficios de Rendimiento Empresarial**

### ⚡ **Optimización del Rendimiento Total**
- **Ejecución Paralela**: Múltiples agentes trabajando simultáneamente
- **Uso de Recursos**: Máxima eficiencia de la capacidad disponible del modelo de IA
- **Reducción de Tiempo**: Disminución significativa del tiempo total de procesamiento
- **Arquitectura Escalable**: Añade fácilmente más agentes concurrentes según sea necesario

### 🛡️ **Confiabilidad y Resiliencia**
- **Tolerancia a Fallos**: Fallos individuales de agentes no detienen todo el flujo de trabajo
- **Aislamiento de Errores**: Problemas en una rama concurrente no afectan a las demás
- **Degradación Gradual**: El sistema sigue operando incluso con capacidad reducida de agentes
- **Mecanismos de Recuperación**: Reintento automático y manejo de errores para operaciones fallidas

### 📊 **Monitoreo y Observabilidad**
- **Seguimiento de Ejecución Concurrente**: Monitorea el progreso de todas las operaciones paralelas
- **Métricas de Rendimiento**: Mide la aceleración y las ganancias en eficiencia
- **Análisis de Uso de Recursos**: Optimiza la asignación de agentes concurrentes
- **Identificación de Cuellos de Botella**: Detecta y resuelve limitaciones de rendimiento

¡Construyamos flujos de trabajo de IA concurrentes de alto rendimiento! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
import os
from typing import Any

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowViz,
    handler,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

In [ ]:
ResearcherAgentName = "Researcher-Agent"
ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction."

In [ ]:
PlanAgentName = "Plan-Agent"
PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings."

In [ ]:
research_agent = await provider.create_agent(
    name=ResearcherAgentName,
    instructions=ResearcherAgentInstructions,
)

plan_agent = await provider.create_agent(
    name=PlanAgentName,
    instructions=PlanAgentInstructions,
)

In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [research_agent, plan_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
events = await workflow.run("Plan a trip to Seattle in December")
outputs = events.get_outputs()

In [ ]:
if outputs:
    print("===== Final Aggregated Responses =====")
    # outputs is a list of AgentResponse objects, one per output executor
    # (research_agent then plan_agent), in the order given to output_executors.
    for i, response in enumerate(outputs, start=1):
        print(f"{'-' * 60}\n\n{i:02d}:\n{response.text}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional humana. No somos responsables de cualquier malentendido o interpretación errónea que surja del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
